In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, f1_score,
                             precision_score, recall_score,
                             average_precision_score,
                             roc_auc_score)
import json

In [3]:
SEED = 4266
DATA_DIR = "../../data/processed"
# DROP_OUT = 0.3
# LEARNING_RATE = 1e-3
# HIDDEN_DIM = 100
INPUT_DIM = 59
OUTPUT_DIM = 5
# LAYER_DIM = 1
EPOCHS = 20
PATIENCE = 4
MIN_DELTA = 1e-4

#hyperparameter search space
HIDDEN_DIM_CHOICES = [32, 64, 100, 128, 256]
LAYER_DIM_MIN = 1
LAYER_DIM_MAX = 2
DROP_OUT_MIN = 0.0
DROP_OUT_MAX = 0.5
DROP_OUT_STEP = 0.1
LEARNING_RATE_MIN = 1e-4
LEARNING_RATE_MAX = 5e-3
WEIGHT_DECAY_MIN = 1e-6
WEIGHT_DECAY_MAX = 1e-3
OPTIMIZER_CHOICES = ["adam", "adamw", "rmsprop"]
OPTIMIZER_MAP = {
    "adam": torch.optim.Adam,
    "adamw": torch.optim.AdamW,
    "rmsprop": torch.optim.RMSprop
}
BATCH_SIZE_CHOICES = [128, 256, 512]

OBJECTIVE_N_TRIALS = 30

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# feats
X_train_raw = np.load(f"{DATA_DIR}/X_train.npy")
X_val_raw   = np.load(f"{DATA_DIR}/X_val.npy")
X_test_raw  = np.load(f"{DATA_DIR}/X_test.npy")

# src-des
# pairs_train_raw = np.load(f"{DATA_DIR}/pairs_train.npy", allow_pickle=True)
# pairs_val_raw   = np.load(f"{DATA_DIR}/pairs_val.npy", allow_pickle=True)
# pairs_test  = np.load(f"{DATA_DIR}/pairs_test.npy", allow_pickle=True)

# labels
y_train_raw = np.load(f"{DATA_DIR}/y_train.npy")
y_val_raw   = np.load(f"{DATA_DIR}/y_val.npy")
y_test_raw  = np.load(f"{DATA_DIR}/y_test.npy")


In [6]:
X_train = torch.tensor(X_train_raw, dtype=torch.float32)
y_train = torch.tensor(y_train_raw, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test_raw, dtype=torch.float32)
y_test = torch.tensor(y_test_raw, dtype=torch.float32)
# y_test = torch.tensor(y_test, dtype=torch.long)
X_val = torch.tensor(X_val_raw, dtype=torch.float32)
y_val = torch.tensor(y_val_raw, dtype=torch.float32)
# y_val = torch.tensor(y_val, dtype=torch.long)

y_train_t = torch.tensor(np.argmax(y_train_raw, axis=1), dtype=torch.long)
y_val_t = torch.tensor(np.argmax(y_val_raw, axis=1), dtype=torch.long)
y_test_t = torch.tensor(np.argmax(y_test_raw, axis=1), dtype=torch.long)

In [7]:
print("shape:", y_train.shape)
print("dtype:", y_train.dtype)
print("ndim: ", y_train.ndim)
print("min/max:", y_train.min().item(), y_train.max().item())

shape: torch.Size([165770, 5])
dtype: torch.float32
ndim:  2
min/max: 0.0 1.0


In [8]:
class_counts = torch.bincount(y_train_t, minlength=OUTPUT_DIM).float()
# class_counts = y_train.sum(dim=0).float()
weight = 1.0 / class_counts
weight = weight / weight.sum() * OUTPUT_DIM
print(weight)

tensor([1.4402e+00, 1.1103e+00, 6.2498e-01, 1.8233e+00, 1.2043e-03])


In [9]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim  = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim,
                            batch_first=True,
                            dropout=dropout if layer_dim > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.attn    = nn.Linear(hidden_dim, 1)
        self.fc      = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        #we reset the hidden state for each batch of sequences so that they are independent of each other. This is important for training on batches of sequences.
        out, _ = self.lstm(x)                       # (B, T, H) — zeros state by default
        # attention pooling over time
        scores  = self.attn(out).squeeze(-1)         # (B, T)
        #weight of each timestep
        weights = torch.softmax(scores, dim=1)       # (B, T)
        #weighted average
        pooled  = (out * weights.unsqueeze(-1)).sum(1)  # (B, H)
        #regularisation
        pooled  = self.dropout(pooled)
        return self.fc(pooled)

In [ ]:
def baseline():
    hidden_dim = 128
    layer_dim = 2
    dropout = 0.2
    lr = 1e-4
    weight_decay = 1e-4
    epochs = 30
    seed = SEED
    patience = 5
    optimizer_name = "adam"
    batch_size = 256
    min_delta = 0

    #set seed for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)

    #pair x and y, shuffle the data, and create batches for training
    loader = DataLoader(
        TensorDataset(X_train, y_train_t),
        batch_size=batch_size, shuffle=True, drop_last=False)

    #create LSTM
    model = LSTMModel(INPUT_DIM, hidden_dim, layer_dim, OUTPUT_DIM, dropout).to(device)
    #calculate cross entropy loss with class weight for multiclass
    criterion = nn.CrossEntropyLoss(weight=weight.to(device))

    #dict to map to library optimizers
    OptimCls = OPTIMIZER_MAP[optimizer_name]
    optimizer = OptimCls(model.parameters(), lr=lr, weight_decay=weight_decay)

    #move to gpu
    X_val_dev = X_val.to(device)
    #ensure on cpu
    y_val_np  = y_val_t.cpu().numpy()

    #trackers for metric, state and early stop
    best_ap, best_state, no_improve = -1.0, None, 0

    #history for plotting
    history = {
        "epoch": [],
        "train_loss": [],
        "val_average_precision": [],
        "best_average_precision": [],
        "no_improve": [],
    }

    #run optimising loop
    for epoch in range(epochs):
        print("epoch ", epoch)
        #model training
        model.train()
        #get stats for history
        epoch_loss, n_batches = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            #cap gradient at 1
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            #weight update
            optimizer.step()

            #update stats
            epoch_loss += loss.item()
            n_batches += 1

        #model evaluation
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_dev)
            val_probs = torch.softmax(val_logits, dim=1).cpu().numpy()

        y_val_onehot = np.eye(OUTPUT_DIM)[y_val_np]
        val_average_precision = average_precision_score(
            y_val_onehot, val_probs, average='macro')

        
        history["epoch"].append(epoch)
        history["train_loss"].append(epoch_loss / n_batches)
        history["val_average_precision"].append(val_average_precision)
        history["best_average_precision"].append(max(best_ap, val_average_precision))
        history["no_improve"].append(no_improve)

        #early stop with patience
        if val_average_precision > best_ap + min_delta:
            best_ap = val_average_precision
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_ap, model, history

In [ ]:
average_precision, model, history = baseline()

epoch  0
epoch  1
epoch  2
epoch  3


In [25]:
print(average_precision)
print(history)

0.9995238091640012
{'epoch': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'train_loss': [0.24292211054555846, 0.06845817425448646, 0.07507874240291705, 0.057396294506552537, 0.05332361464274411, 0.04840191034199217, 0.03962985466882789, 0.03062652683770009, 0.037856518021723795, 0.036272406227478717], 'val_average_precision': [0.9929329070781652, 0.9937928746231576, 0.9980237756485781, 0.9939455313810661, 0.9995238091640012, 0.9970026709189496, 0.9980238077250727, 0.9980238080847588, 0.9980238091640011, 0.9980158726560646], 'best_average_precision': [0.9929329070781652, 0.9937928746231576, 0.9980237756485781, 0.9980237756485781, 0.9995238091640012, 0.9995238091640012, 0.9995238091640012, 0.9995238091640012, 0.9995238091640012, 0.9995238091640012], 'no_improve': [0, 0, 0, 0, 1, 0, 1, 2, 3, 4]}


In [26]:
print(history["val_average_precision"])

[0.9929329070781652, 0.9937928746231576, 0.9980237756485781, 0.9939455313810661, 0.9995238091640012, 0.9970026709189496, 0.9980238077250727, 0.9980238080847588, 0.9980238091640011, 0.9980158726560646]


In [27]:
params = list(model.parameters())
print(params)

[Parameter containing:
tensor([[-0.0224,  0.0278,  0.0567,  ..., -0.0389, -0.0674,  0.3241],
        [ 0.0298,  0.0006,  0.0812,  ...,  0.1244, -0.0485, -0.1767],
        [ 0.0874, -0.0658, -0.0555,  ...,  0.0267, -0.0894,  0.2396],
        ...,
        [ 0.0513, -0.0155,  0.0031,  ...,  0.2559, -0.1348,  0.1510],
        [-0.1533, -0.0656,  0.0114,  ..., -0.1527, -0.0532,  0.1068],
        [-0.0009, -0.2111, -0.1827,  ...,  0.0072, -0.1023,  0.0244]],
       requires_grad=True), Parameter containing:
tensor([[ 0.1149, -0.0281,  0.1174,  ..., -0.2710, -0.0528,  0.0597],
        [ 0.2001,  0.0306,  0.1529,  ..., -0.0683, -0.0871,  0.0469],
        [ 0.0224,  0.0128,  0.1031,  ..., -0.1233, -0.0207,  0.1870],
        ...,
        [ 0.1380, -0.0195,  0.1529,  ..., -0.1215, -0.0392,  0.0655],
        [ 0.1700,  0.0318,  0.0782,  ..., -0.1545, -0.1498,  0.1709],
        [ 0.0951,  0.0280,  0.1790,  ..., -0.0922, -0.0951,  0.0157]],
       requires_grad=True), Parameter containing:
tensor([ 

In [28]:
torch.save(model.state_dict(), "lstm_model.pt")

In [29]:
def compute_metrics(probs, preds, labels, num_classes=5):
    # multiclass need to use 1 hot
    labels_onehot = np.eye(num_classes)[labels]

    return {
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "pr_auc": average_precision_score(labels_onehot, probs, average="macro"),
        "report": classification_report(
            labels,
            preds,
            target_names=["Backdoor", "CommInj", "DoS", "Reconn", "normal"],
            digits=4,
            zero_division=0
        )
    }
def evaluate_lstm(model, dataloader, loss_fn, device, num_classes=5):
    model.eval()
    val_loss = 0
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        # No src, dst, or edge_feat for standard sequence data
        for batch_x, y in dataloader:
            batch_x, y = batch_x.to(device), y.to(device)

            # 2. CHANGE: Forward pass only takes the sequence batch
            logits = model(batch_x)
            
            val_loss += loss_fn(logits, y).item()

            # prob and pred for each batch (This part stays exactly the same)
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_labels.append(y.cpu().numpy())

    # flatten into 1 array (This part stays exactly the same)
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    # Compute metrics (This part stays exactly the same)
    metrics = compute_metrics(probs, preds, labels, num_classes)
    metrics["loss"] = val_loss / len(dataloader)

    return metrics

In [30]:
test_metrics = evaluate_lstm(model, DataLoader(TensorDataset(X_test, y_test_t), batch_size=128), nn.CrossEntropyLoss(weight=weight.to(device)), device)
print(
    f"Test Loss: {test_metrics['loss']:.4f} | "
    f"F1: {test_metrics['f1']:.4f} | "
    f"Precision: {test_metrics['precision']:.4f} | "
    f"Recall: {test_metrics['recall']:.4f} | "
    f"PR-AUC: {test_metrics['pr_auc']:.4f}"
)

print("\nClassification Report:")
print(test_metrics["report"])

Test Loss: 0.1218 | F1: 0.8749 | Precision: 0.8485 | Recall: 0.9368 | PR-AUC: 0.9414

Classification Report:
              precision    recall  f1-score   support

    Backdoor     0.3333    0.7692    0.4651        39
     CommInj     0.9091    0.9804    0.9434        51
         DoS     1.0000    1.0000    1.0000        91
      Reconn     1.0000    0.9355    0.9667        31
      normal     0.9999    0.9988    0.9993     47152

    accuracy                         0.9985     47364
   macro avg     0.8485    0.9368    0.8749     47364
weighted avg     0.9992    0.9985    0.9988     47364



In [31]:
def load_baseline():
    hidden_dim = 128
    layer_dim = 2
    dropout = 0.2
    lr = 1e-3
    weight_decay = 0
    epochs = 100
    seed = SEED
    patience = 10
    optimizer_name = "adam"
    batch_size = 128
    min_delta = 0

    #set seed for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)


    #create LSTM
    model = LSTMModel(INPUT_DIM, hidden_dim, layer_dim, OUTPUT_DIM, dropout).to(device)
    model.load_state_dict(torch.load("lstm_model.pt", map_location=device))##, weights_only=True))

    return model

In [18]:
baseline_model = load_baseline()

In [19]:
%pip install shap


  Using cached shap-0.54.1-cp312-abi3-win_amd64.whl.metadata (23 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
Using cached shap-0.54.1-cp312-abi3-win_amd64.whl (751 kB)
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)

   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
